# A single confidence for a hard judge prediction

Give the selected class $\alpha$ and each other class $(1-\alpha)/2$.
If the hard prediction has accuracy $a$, its mean log loss is

$$L(\alpha)=-a\ln\alpha-(1-a)\ln((1-\alpha)/2).$$

The minimum occurs at $\alpha=a$. These calculations reuse the frozen Gemma judge's
512-row, 2,048-token experiment; hard predictions are the argmax of saved label
probabilities, without new inference. Lower log loss is better.


In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.special import xlogy
from IPython.display import display
repo=Path.cwd()
if repo.name=='notebooks':repo=repo.parent
result=json.loads((repo/'results/attempt_003/fixed_alpha_2048.json').read_text())
table=pd.DataFrame(result['rows'])
display(table[['view','rows','correct','accuracy','optimal_alpha_fit_same_rows',
               'other_class_probability','best_same_rows_log_loss','cross_fitted_log_loss']])


,view,rows,correct,accuracy,optimal_alpha_fit_same_rows,other_class_probability,best_same_rows_log_loss,cross_fitted_log_loss
0,original,512,275,0.537109,0.537109,0.231445,1.011242,1.013872
1,average,512,261,0.509766,0.509766,0.245117,1.032761,1.037038


The fitted optimum uses all 512 labels. The cross-fitted estimate fits alpha
on four folds and scores the remaining fold, using five prompt-group folds.
These are local development estimates, not official Kaggle scores. Development
has also been used in previous experiments, so this is not untouched confirmation.

Verify the analytic optimum and the accuracy required for a 0.8 loss:


In [2]:
def best_loss(a):
    return float(-xlogy(a,a)-xlogy(1-a,(1-a)/2))
for row in result['rows']:
    a=row['accuracy']
    np.testing.assert_allclose(best_loss(a),row['best_same_rows_log_loss'])
    grid=np.linspace(a-.01,a+.01,2001)
    objective=-a*np.log(grid)-(1-a)*np.log((1-grid)/2)
    np.testing.assert_allclose(grid[np.argmin(objective)],a,atol=1e-5)
    print(row['view'],'alpha:',a,'fitted loss:',best_loss(a),
          'cross-fitted loss:',row['cross_fitted_log_loss'])
required=brentq(lambda a:best_loss(a)-.8,1/3+1e-8,1-1e-8)
np.testing.assert_allclose(required,result['accuracy_for_log_loss_0_8_above_chance'])
print(f'Accuracy for 0.8 loss on the above-chance branch: {required:.4%}')
print('Original-order cross-fitted descriptive 95% group-bootstrap interval:',
      result['rows'][0]['cross_fitted_log_loss_95_interval'])


original alpha: 0.537109375 fitted loss: 1.0112417666125035 cross-fitted loss: 1.013872133031994
average alpha: 0.509765625 fitted loss: 1.032761008413034 cross-fitted loss: 1.0370383556109535
Accuracy for 0.8 loss on the above-chance branch: 71.1986%
Original-order cross-fitted descriptive 95% group-bootstrap interval: [0.9774627173386599, 1.0496083713938693]


This improves on the original-order raw log loss **1.42159**, but is worse
than temperature plus a tie intercept (**0.98076**, same five-fold protocol).
Assigning the same confidence to every prediction discards differences in
confidence and tie preference. Approximately **71.2% accuracy** is needed to
reach 0.8 with this rule, compared with the judge's present **53.7%**.

Reproduce with `scripts/analyze_fixed_alpha.py`; saved configuration and label
alignment are checked. Only aggregate JSON and this notebook are committed.
Prediction arrays and group IDs remain under `~/database`. The ongoing judge
submission configuration was not changed. See `docs/FIXED_ALPHA.md` for details.
